In [ ]:
import os

import h5py
import numpy as np
import pandas as pd

from mlpng.generator import Generator
from mlpng.utils import plot_predictions

In [ ]:
NSIMS = 1000
# FNL_MIN, FNL_MAX = -1000.0, 1000.0

NSIDE_CONFIGS = {
    64: ("settings/n64.json", "data/data/l191_n64_T_10000_p1.0.hdf5"),
    128: ("settings/n128.json", "data/data/l383_n128_T_10000_p1.0.hdf5"),
    256: ("settings/n256.json", "data/data/l767_n256_T_10000_p1.0.hdf5"),
}

SHAPE_SETS = {
    "local": ["local"],
    # "all":   ["local", "equilateral", "orthogonal"], # not supported
}

LENSING_OPTIONS = [False, True]

In [ ]:
generators = {}
for nside, (settings_file, _) in NSIDE_CONFIGS.items():
    generators[nside] = Generator(
        argv=[settings_file, "--nsims", "10000", "--lensing", "--shapes", "all"]
    )

In [ ]:
def sigma_stats(residuals, sigma):
    empirical_sigma = np.std(residuals)
    print(
        f"  Fisher σ: {sigma:.4f},  Empirical σ: {empirical_sigma:.4f}  (ratio: {empirical_sigma / sigma:.3f})"
    )
    expected = {1: 68.27, 2: 95.45, 3: 99.73}
    for n, exp in expected.items():
        within = np.sum(np.abs(residuals) < n * sigma)
        print(f"  {n}σ: {100 * within / len(residuals):.1f}% (expected {exp:.1f}%)")

In [ ]:
import matplotlib.pyplot as plt

phi = np.array([1, 2, 5, 10, 20, 50, 100, 500, 1000])
sigma_64 = np.array([59, 62, 65, 66, 75, 85, 90, 126, 164])
sigma_128 = np.array([37, 37, 38, 37, 54, 60, 61, 117, 130])
sigma_256 = np.array([64, 42, 57, 65, 42, 57, 65, 71, 94])

# Initialize error arrays (to be populated by error_finder.sh results - Full Range)
sigma_err_64 = np.full(len(phi), 48)
sigma_err_128 = np.full(len(phi), 22)
sigma_err_256 = np.full(len(phi), 11)

# Create mapping dict for easy reference
sigma_err = {64: sigma_err_64, 128: sigma_err_128, 256: sigma_err_256}

# Plot: Sigma vs Phi with error bars (Full Range)
fig, ax = plt.subplots(figsize=(12, 7))
ax.errorbar(
    phi,
    sigma_64,
    yerr=sigma_err_64,
    marker="o",
    linewidth=2,
    markersize=6,
    label="nside=64",
    capsize=5,
    capthick=2,
)
ax.errorbar(
    phi,
    sigma_128,
    yerr=sigma_err_128,
    marker="s",
    linewidth=2,
    markersize=6,
    label="nside=128",
    capsize=5,
    capthick=2,
)
ax.errorbar(
    phi,
    sigma_256,
    yerr=sigma_err_256,
    marker="^",
    linewidth=2,
    markersize=6,
    label="nside=256",
    capsize=5,
    capthick=2,
)
ax.set_xlabel(r"$A_\mathrm{lens}$", fontsize=12)
ax.set_ylabel(r"$\sigma(f_\mathrm{NL}^\mathrm{loc})$", fontsize=12)
ax.set_xscale("log")
ax.grid(True, alpha=0.3)
ax.legend(loc="best", fontsize=11)
# ax.set_title("Sigma vs Phi - Full Range (with error bands)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Reduced Range [-100,100] Section
# To be populated by user with: bash error_finder.sh <nside> <phi_scale> --reduced

# Initialize reduced range arrays (to be populated manually by user)
phi = np.array([1, 2, 5, 10, 20, 50, 100, 500])  # , 1000])
sigma_reduced_64 = np.array([49, 53, 52, 59, 58, 70, 77, 113])  # , 160])
sigma_reduced_128 = np.array([46, 33, 36, 37, 42, 50, 63, 97])  # , 114])
sigma_reduced_256 = np.array([23, 19, 32, 34, 30, 35, 40, 61])  # , 83])

# Initialize reduced range error arrays
sigma_err_reduced_64 = np.full(len(phi), np.nan)
sigma_err_reduced_128 = np.full(len(phi), np.nan)
sigma_err_reduced_256 = np.full(len(phi), np.nan)

# Create mapping dict for easy reference
sigma_err_reduced = {
    64: sigma_err_reduced_64,
    128: sigma_err_reduced_128,
    256: sigma_err_reduced_256,
}

# Plot: Sigma vs Phi with error bars (Reduced Range)
fig, ax = plt.subplots(figsize=(12, 7))
ax.errorbar(
    phi,
    sigma_reduced_64,
    yerr=sigma_err_reduced_64,
    marker="o",
    linewidth=2,
    markersize=6,
    label="nside=64",
    capsize=5,
    capthick=2,
)
ax.errorbar(
    phi,
    sigma_reduced_128,
    yerr=sigma_err_reduced_128,
    marker="s",
    linewidth=2,
    markersize=6,
    label="nside=128",
    capsize=5,
    capthick=2,
)
ax.errorbar(
    phi,
    sigma_reduced_256,
    yerr=sigma_err_reduced_256,
    marker="^",
    linewidth=2,
    markersize=6,
    label="nside=256",
    capsize=5,
    capthick=2,
)
ax.set_xlabel("phi", fontsize=12)
ax.set_ylabel("sigma", fontsize=12)
ax.set_xscale("log")
ax.grid(True, alpha=0.3)
ax.legend(loc="best", fontsize=11)
ax.set_title("Sigma vs Phi - Reduced Range [-100,100] (with error bands)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Reduced ratio plot with R_GL baseline overlay
from scipy.interpolate import interp1d

nsides = [64, 128, 256]
markers = {64: "o", 128: "s", 256: "^"}
colors = {64: "tab:blue", 128: "tab:orange", 256: "tab:green"}

sigma_reduced = {
    64: sigma_reduced_64,
    128: sigma_reduced_128,
    256: sigma_reduced_256,
}

# Use full-range errors only where reduced-range errors are still missing.
sigma_err_reduced_plot = {
    64: np.where(
        np.isnan(sigma_err_reduced_64), sigma_err_64[: len(phi)], sigma_err_reduced_64
    ),
    128: np.where(
        np.isnan(sigma_err_reduced_128),
        sigma_err_128[: len(phi)],
        sigma_err_reduced_128,
    ),
    256: np.where(
        np.isnan(sigma_err_reduced_256),
        sigma_err_256[: len(phi)],
        sigma_err_reduced_256,
    ),
}

baseline_phi = np.array(
    [1, 2, 3, 4, 5, 10, 20, 50, 100, 150, 200, 250, 300, 400, 500],
    dtype=float,
)
baseline = {
    64: np.array(
        [
            0.9986,
            0.9975,
            0.9956,
            0.9930,
            0.9896,
            0.9631,
            0.8748,
            0.5867,
            0.3405,
            0.2349,
            0.1784,
            0.1434,
            0.1199,
            0.0902,
            0.0723,
        ],
        dtype=float,
    ),
    128: np.array(
        [
            0.9989,
            0.9933,
            0.9843,
            0.9721,
            0.9570,
            0.8538,
            0.6332,
            0.3108,
            0.1616,
            0.1084,
            0.0813,
            0.0652,
            0.0544,
            0.0408,
            0.0327,
        ],
        dtype=float,
    ),
    256: np.array(
        [
            0.9955,
            0.9877,
            0.9751,
            0.9587,
            0.9380,
            0.8077,
            0.5650,
            0.2649,
            0.1360,
            0.0912,
            0.0687,
            0.0549,
            0.0456,
            0.0344,
            0.0274,
        ],
        dtype=float,
    ),
}

# Interpolate baseline to measurement phi points
baseline_interp = {}
for nside in nsides:
    f = interp1d(baseline_phi, baseline[nside], kind="cubic", fill_value="extrapolate")
    baseline_interp[nside] = f(phi)

fig, ax = plt.subplots(figsize=(8, 6))

for nside in nsides:
    # Plot sigma_err_reduced_plot / baseline
    with np.errstate(divide="ignore", invalid="ignore"):
        ratio = np.where(
            baseline_interp[nside] != 0,
            sigma_err_reduced_plot[nside] / baseline_interp[nside],
            np.nan,
        )

    (line1,) = ax.plot(
        phi,
        ratio,
        f"{markers[nside]}-.",
        color=colors[nside],
        linewidth=2,
        markersize=6,
        label=rf"$N_{{\mathrm{{side}}}} = {nside}$, Estimator",
    )

    # Plot sigma_reduced
    (line2,) = ax.plot(
        phi,
        sigma_reduced[nside],
        f"{markers[nside]}-",
        color=colors[nside],
        linewidth=2,
        markersize=5,
        alpha=0.7,
        label=rf"$N_{{\mathrm{{side}}}} = {nside}, ML$",
    )

ax.axvline(25, color="gray", linestyle="--", linewidth=1)

ax.set_xlabel(r"$A_\mathrm{lens}$", fontsize=12)
ax.set_ylabel(r"$\sigma( f_{\mathrm{NL}})$", fontsize=12)
ax.set_xscale("log")
ax.set_yscale("log")
ax.grid(True, alpha=0.3)
ax.legend(loc="best", fontsize=10)
# ax.set_title(
#     r"Error Ratio and Measured $\sigma$ vs $A_\mathrm{lens}$",
#     fontsize=13,
# )
plt.tight_layout()
plt.show()

In [ ]:
# Non-reduced ratio plot with R_GL baseline overlay
phi_full = np.array([1, 2, 5, 10, 20, 50, 100, 500, 1000], dtype=float)
sigma_full = {
    64: sigma_64,
    128: sigma_128,
    256: sigma_256,
}
sigma_err_full = {
    64: sigma_err_64,
    128: sigma_err_128,
    256: sigma_err_256,
}

baseline_phi_full = np.array(
    [1, 2, 3, 4, 5, 10, 20, 50, 100, 150, 200, 250, 300, 400, 500],
    dtype=float,
)
baseline_full = {
    64: np.array(
        [
            0.9986,
            0.9975,
            0.9956,
            0.9930,
            0.9896,
            0.9631,
            0.8748,
            0.5867,
            0.3405,
            0.2349,
            0.1784,
            0.1434,
            0.1199,
            0.0902,
            0.0723,
        ],
        dtype=float,
    ),
    128: np.array(
        [
            0.9989,
            0.9933,
            0.9843,
            0.9721,
            0.9570,
            0.8538,
            0.6332,
            0.3108,
            0.1616,
            0.1084,
            0.0813,
            0.0652,
            0.0544,
            0.0408,
            0.0327,
        ],
        dtype=float,
    ),
    256: np.array(
        [
            0.9955,
            0.9877,
            0.9751,
            0.9587,
            0.9380,
            0.8077,
            0.5650,
            0.2649,
            0.1360,
            0.0912,
            0.0687,
            0.0549,
            0.0456,
            0.0344,
            0.0274,
        ],
        dtype=float,
    ),
}

fig, ax = plt.subplots(figsize=(12, 7))

for nside in nsides:
    with np.errstate(divide="ignore", invalid="ignore"):
        ratio_full = np.where(
            sigma_full[nside] != 0,
            sigma_err_full[nside] / sigma_full[nside],
            np.nan,
        )

    (line,) = ax.plot(
        phi_full,
        ratio_full,
        f"{markers[nside]}-",
        color=colors[nside],
        linewidth=2,
        markersize=6,
    )
    ax.plot(
        baseline_phi_full,
        baseline_full[nside],
        f"{markers[nside]}--",
        color=colors[nside],
        linewidth=2,
        markersize=5,
        alpha=0.9,
    )
    line.set_label(rf"$N_{{\mathrm{{side}}}} = {nside}$")

ax.set_xlabel(r"$A_\mathrm{lens}$", fontsize=12)
ax.set_ylabel(r"$\sigma(f_{\mathrm{NL}})$", fontsize=12)
ax.set_xscale("log")
ax.grid(True, alpha=0.3)
ax.legend(loc="best", fontsize=11)
ax.set_title(
    r"Non-Reduced $\sigma(f_{\mathrm{NL}}^{\mathrm{ML}}) / \sigma(f_{\mathrm{NL}}^{\mathrm{Fisher}})$",
    fontsize=13,
)
plt.tight_layout()
plt.show()

In [ ]:
stop

In [ ]:
rng = np.random.default_rng(42)
results = []

for nside, (_, data_file) in NSIDE_CONFIGS.items():
    gen = generators[nside]
    pol_idxs = gen.pol_idxs()
    fnl_true = rng.uniform(-1000.0, 1000.0, (NSIMS, 1, 1))

    for lensed in LENSING_OPTIONS:
        l_str = "lensed" if lensed else "unlensed"

        for set_name, shapes in SHAPE_SETS.items():
            for shape in shapes:
                label = f"{l_str} {shape}"
                print(f"\n=== {label} ===")

                with h5py.File(data_file, "r") as f:
                    alm_l = f[f"alm_l/{l_str}/{shape}"][:][:NSIMS]
                    alm_nl = f[f"alm_nl/{l_str}/{shape}"][:][:NSIMS]

                if not lensed:
                    alm_l = alm_l[:, pol_idxs]

                alm = alm_l + fnl_true * alm_nl

                ksw = gen.get_ksw(shape, lensed=lensed)
                fisher = ksw.compute_fisher()
                sigma = 1.0 / np.sqrt(fisher)
                print(f"  Fisher: {fisher:.6f},  \u03c3: {sigma:.4f}")

                estimates, *_ = ksw.compute_estimate_batch(
                    lambda idx, _alm=alm, _lensed=lensed: gen.icov_func(
                        _alm[idx], lensed=_lensed
                    ),
                    range(NSIMS),
                    # range(100),
                    fisher=fisher,
                )
                estimates = estimates.T.flatten()
                fnl_flat = fnl_true.flatten()
                residuals = estimates - fnl_flat

                sigma_stats(residuals, sigma)

                plot_predictions(
                    fnl_flat,
                    estimates,
                    sigma=sigma,
                    title=f"KSW predictions for {label} (σ={np.std(residuals):.4f})",
                    save=False,
                    show=True,
                    close=True,
                    legend=False,
                )

                results.append(
                    {
                        "nside": nside,
                        "shape": shape,
                        "lensed": l_str,
                        "set": set_name,
                        "sigma": round(float(sigma), 4),
                        "mean_resid": round(float(np.mean(residuals)), 4),
                        "std_resid": round(float(np.std(residuals)), 4),
                    }
                )

display(pd.DataFrame(results))

In [ ]:
rng = np.random.default_rng(42)
results = []

for nside, (_, data_file) in NSIDE_CONFIGS.items():
    gen = generators[nside]
    pol_idxs = gen.pol_idxs()
    fnl_true = rng.uniform(-100.0, 100.0, (NSIMS, 1, 1))

    for lensed in LENSING_OPTIONS:
        l_str = "lensed" if lensed else "unlensed"

        for set_name, shapes in SHAPE_SETS.items():
            for shape in shapes:
                label = f"{l_str} {shape}"
                print(f"\n=== {label} ===")

                with h5py.File(data_file, "r") as f:
                    alm_l = f[f"alm_l/{l_str}/{shape}"][:][:NSIMS]
                    alm_nl = f[f"alm_nl/{l_str}/{shape}"][:][:NSIMS]

                if not lensed:
                    alm_l = alm_l[:, pol_idxs]

                alm = alm_l + fnl_true * alm_nl

                ksw = gen.get_ksw(shape, lensed=lensed)
                fisher = ksw.compute_fisher()
                sigma = 1.0 / np.sqrt(fisher)
                print(f"  Fisher: {fisher:.6f},  \u03c3: {sigma:.4f}")

                estimates, *_ = ksw.compute_estimate_batch(
                    lambda idx, _alm=alm, _lensed=lensed: gen.icov_func(
                        _alm[idx], lensed=_lensed
                    ),
                    range(NSIMS),
                    fisher=fisher,
                )
                estimates = estimates.T.flatten()
                fnl_flat = fnl_true.flatten()
                residuals = estimates - fnl_flat

                sigma_stats(residuals, sigma)

                plot_predictions(
                    fnl_flat,
                    estimates,
                    sigma=sigma,
                    title=f"KSW predictions for {label} (σ={np.std(residuals):.4f})",
                    save=False,
                    show=True,
                    close=True,
                )

                results.append(
                    {
                        "nside": nside,
                        "shape": shape,
                        "lensed": l_str,
                        "set": set_name,
                        "sigma": round(float(sigma), 4),
                        "mean_resid": round(float(np.mean(residuals)), 4),
                        "std_resid": round(float(np.std(residuals)), 4),
                    }
                )

display(pd.DataFrame(results))

In [ ]:
import io
from contextlib import redirect_stdout
import lenspyx
import numpy as np
import healpy as hp

stop

phis = [1, 2, 10, 20, 50, 100, 300, 1000]

rng = np.random.default_rng(42)
results = []

for phi in phis:
    for nside, (_, data_file) in NSIDE_CONFIGS.items():
        gen = generators[nside]
        pol_idxs = gen.pol_idxs()
        fnl_true = rng.uniform(-1000.0, 1000.0, (NSIMS, 1, 1))

        lmax_len = len(gen.ells)
        fl = np.arange(lmax_len + 1) * np.arange(1, lmax_len + 2)
        fl = np.sqrt(fl)
        geom_info = ("healpix", {"nside": gen.nside})

        for lensed in [True]:
            l_str = "lensed" if lensed else "unlensed"

            for set_name, shapes in SHAPE_SETS.items():
                for shape in shapes:
                    label = f"{l_str} {shape} (φ={phi})"
                    print(f"\n=== {label} ===")

                    with h5py.File(data_file, "r") as f:
                        alm_l = f[f"alm_l/unlensed/{shape}"][:][:NSIMS]
                        alm_nl = f[f"alm_nl/unlensed/{shape}"][:][:NSIMS]
                        alm_phi_all = f["alm_phi"][:][:NSIMS]

                    # Process lensing for each simulation
                    alm_lensed_list = []
                    with redirect_stdout(io.StringIO()):
                        for sim_idx in range(NSIMS):
                            fnl_vals = fnl_true[sim_idx].flatten()

                            alm = alm_l[sim_idx].astype(np.complex128) + np.einsum(
                                "i...,i...->...", fnl_vals, alm_nl[sim_idx]
                            )
                            alm = np.asarray(alm, dtype=np.complex128)
                            alm_phi = alm_phi_all[sim_idx] * phi

                            # Apply deflection field
                            dlm = hp.almxfl(alm_phi, fl)

                            # Lens the alm
                            lenmap = lenspyx.alm2lenmap(
                                alm,
                                dlm,
                                geometry=geom_info,
                                nthreads=1,
                            )

                            # Convert lensed map back to alm
                            lensed_alm = hp.map2alm(
                                lenmap, lmax=gen.lmax, use_pixel_weights=True
                            )
                            alm_lensed_list.append(lensed_alm)

                    # Stack lensed alms: shape should be (NSIMS, npol, nalm)
                    alm_combined = np.array(alm_lensed_list, dtype=np.complex128)

                    ksw = gen.get_ksw(shape, lensed=lensed)
                    fisher = ksw.compute_fisher()
                    sigma = 1.0 / np.sqrt(fisher)

                    estimates, *_ = ksw.compute_estimate_batch(
                        lambda idx, _alm=alm_combined, _lensed=lensed: gen.icov_func(
                            _alm[idx, [0]], lensed=_lensed
                        ),
                        range(NSIMS),
                        fisher=fisher,
                    )
                    estimates = estimates.T.flatten()
                    fnl_flat = fnl_true.flatten()
                    residuals = estimates - fnl_flat
                    sem_resid = np.std(residuals) / np.sqrt(len(residuals))
                    plot_predictions(
                        fnl_flat,
                        estimates,
                        sigma=sigma,
                        title=f"KSW predictions for {label} σ={np.std(residuals):.4f} (SEM={sem_resid:.4f})",
                        save=False,
                        show=True,
                        close=True,
                        legend=False,
                    )

                    results.append(
                        {
                            "phi": phi,
                            "nside": nside,
                            "shape": shape,
                            "lensed": l_str,
                            "set": set_name,
                            "sigma": round(float(sigma), 4),
                            "mean_resid": round(float(np.mean(residuals)), 4),
                            "std_resid": round(float(np.std(residuals)), 4),
                            "sem_resid": round(float(sem_resid), 4),
                        }
                    )

display(pd.DataFrame(results))